# Qwen3-0.6B smoke test on a 2x T4 rig

One short run that exercises the migrated **transformers + PEFT + TRL** backend on a
Turing GPU. T4 has no native bf16, so everything here uses **fp16 + 4-bit** on one
GPU (the second T4 stays idle; dqntrain is single-GPU).

| Check | Stage / helper |
|---|---|
| 2x T4 detection, env report, no bf16 | setup |
| Local parquet data, 8-step data pipeline | `Data(limit=, columns=, filter=, map=, eval=)` |
| SFT in `text`, `chatml`, and `template` formats | 3 x `SFT` |
| User callbacks and checkpoint callbacks | `EventRecorder` |
| DPO on prompt/chosen/rejected | `DPO` |
| GRPO with regex, contains, and a **subprocess-sandboxed** reward | `GRPO` |
| Two adapters, generate, save, switch, disable, delete | `Custom` + `backend` |
| Merge to fp16 and reload | `Merge` + `backend.load_model` |
| Metadata versions in every checkpoint | `meta.json` |
| Kill and resume mid-training | `SMOKE_CRASH=1` |
| Config hash change + `rerun()` | `ConfigChangedError` |
| Fresh vs stale locks | `LockHeldError`, lock takeover |
| Optional lm-eval stage | `RUN_EVAL=1` |
| Optional Hub round-trip | `RUN_HUB=1` |

Expected runtime on one T4: about 5-10 minutes, dominated by the first model
download and 4-bit quantization. Every stage is 1 step except the resume stage
(3 steps). No dataset downloads are needed: all data is generated locally.

## 0. Setup

Installs the tested stack (transformers 5.17, TRL 1.13, PEFT 0.21) and uses the
local `dqntrain` checkout when the notebook is opened from the repository. If Jupyter
starts in another folder, set `DQNTRAIN_SOURCE` to the checkout that contains
`dqntrain/backend.py` (or `pip install --no-deps --force-reinstall /path/to/dqntrain`).
The setup cell fails early when an old dqntrain without `backend.py` is installed.
Restart the kernel if the install cell upgrades packages that were already imported.

Set environment variables before running this cell if needed:

- `SMOKE_GPU=0` / `1` to pick a card (default: last GPU reported by `nvidia-smi`,
  so the second T4 is used on a 2x rig).
- `SKIP_INSTALL=1` to keep the image's package versions.
- `RUN_EVAL=1` to add an `lm-eval` stage (installs `lm-eval` and downloads `arc_easy`).
- `RUN_HUB=1` with `HF_TOKEN` for a tiny private-repo round-trip.
- `SMOKE_TAG=name` to reuse a fixed run name (default: random suffix).

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version as md_version
from importlib.util import find_spec
from pathlib import Path

DQNTRAIN_GIT = "https://github.com/DQN-Labs/dqntrain.git"
SKIP_INSTALL = os.environ.get("SKIP_INSTALL") == "1"
RUN_EVAL = os.environ.get("RUN_EVAL") == "1"
RUN_HUB = os.environ.get("RUN_HUB") == "1"


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def installed_version(name):
    try:
        return md_version(name)
    except Exception:
        return None


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


for name in ("HF_TOKEN", "GITHUB_TOKEN"):
    value = secret(name)
    if value:
        os.environ[name] = value

PINS = {
    "transformers": "5.17.0",
    "trl": "1.13.0",
    "peft": "0.21.2",
    "datasets": "4.8.5",
    "accelerate": "1.15.0",
}
if not SKIP_INSTALL:
    needed = [f"{name}=={pin}" for name, pin in PINS.items() if installed_version(name) != pin]
    for name in ("bitsandbytes", "pyarrow"):
        if find_spec(name) is None:
            needed.append(name)
    if needed:
        print("installing:", ", ".join(needed))
        pip_install(*needed)
    if RUN_EVAL and find_spec("lm_eval") is None:
        pip_install("lm-eval")

for name in [name for name in sys.modules if name == "dqntrain" or name.startswith("dqntrain.")]:
    del sys.modules[name]

shadow = find_spec("dqntrain")
if shadow is not None and shadow.origin is None:
    for location in list(shadow.submodule_search_locations or []):
        parent = str(Path(location).parent)
        print("ignoring namespace-package shadow:", location)
        while parent in sys.path:
            sys.path.remove(parent)

SOURCE = os.environ.get("DQNTRAIN_SOURCE")
candidates = [Path(SOURCE)] if SOURCE else []
candidates += [Path.cwd(), Path.cwd().parent]
SOURCE = next(
    (path for path in candidates if (path / "dqntrain" / "backend.py").is_file()),
    None,
)
if SOURCE is not None:
    sys.path.insert(0, str(SOURCE))
    print("using local dqntrain from", SOURCE.resolve())
elif find_spec("dqntrain.backend") is None:
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        url = f"git+https://x-access-token:{token}@{DQNTRAIN_GIT.removeprefix('https://')}"
    else:
        url = f"git+{DQNTRAIN_GIT}"
    try:
        pip_install("--no-deps", "--force-reinstall", f"dqntrain @ {url}")
    except subprocess.CalledProcessError:
        print("dqntrain install failed: set DQNTRAIN_SOURCE to the repo, or add GITHUB_TOKEN with read access.")
        raise
    for name in [name for name in sys.modules if name == "dqntrain" or name.startswith("dqntrain.")]:
        del sys.modules[name]

if find_spec("dqntrain.backend") is None:
    raise RuntimeError(
        "The installed dqntrain has no dqntrain/backend.py (old Unsloth build).\n"
        "Run this notebook from the dqntrain repo root, or set DQNTRAIN_SOURCE to the folder\n"
        "that contains dqntrain/backend.py, then restart the kernel and rerun from the top.\n"
        "Or install the checkout: pip install --no-deps --force-reinstall /path/to/dqntrain"
    )

import dqntrain as dq
print("dqntrain:", dq.__version__, "from", Path(dq.__file__).resolve())

if os.environ.get("SMOKE_KEEP_ALL") != "1":
    gpu = os.environ.get("SMOKE_GPU")
    if gpu is None:
        visible = [item.strip() for item in os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",") if item.strip()]
        if visible:
            gpu = visible[-1]
        else:
            try:
                gpu = subprocess.run(
                    ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
                    capture_output=True, text=True, check=True,
                ).stdout.split()[-1]
            except Exception:
                gpu = "0"
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("flags: RUN_EVAL =", RUN_EVAL, "| RUN_HUB =", RUN_HUB, "| SKIP_INSTALL =", SKIP_INSTALL)

## 1. Rig and environment

The setup cell pins one GPU. `nvidia-smi` still lists both cards, so this proves the
rig has 2x T4 while the run intentionally uses only one. T4 is compute capability 7.5
and does **not** support bf16; the test fails fast if bf16 is advertised.

In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA GPU is visible. Enable the GPU runtime and restart."
print("torch:", torch.__version__)
print("GPUs visible to torch:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"  cuda:{index} {props.name} {props.total_memory / 1e9:.1f} GB sm_{props.major}{props.minor}")

try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("nvidia-smi:\n" + smi)
except Exception as error:
    print("nvidia-smi unavailable:", error)

DEVICE_NAME = torch.cuda.get_device_name(0)
try:
    BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    BF16 = torch.cuda.is_bf16_supported()
print("selected GPU:", DEVICE_NAME, "| native bf16 supported:", BF16)
if "T4" in DEVICE_NAME:
    assert not BF16, "T4 must not advertise native bf16 support"
    print("Turing T4: fp16 + 4-bit only; bf16 is never used in this notebook")
elif not BF16:
    print("bf16 is unavailable on this GPU; fp16 + 4-bit is used")

import dqntrain as dq

print("dqntrain:", dq.__version__)
dq.env_report()

## 2. Local synthetic data (no dataset downloads)

The data pipeline is fixed: load, shuffle, limit, filter, **then** rename columns,
then map, then column checks, then the optional eval split. The GRPO preview below
uses that order: the filter sees `question` and the map sees the renamed `prompt`.

In [ ]:
import json
import shutil
from pathlib import Path

from datasets import Dataset

ROOT = Path("./smoke-t4-work")
DATA = ROOT / "data"
RUNS = ROOT / "runs"
CACHE = ROOT / "cache"
TAG = os.environ.get("SMOKE_TAG") or os.urandom(3).hex()
print("SMOKE_TAG:", TAG)

if DATA.exists():
    shutil.rmtree(DATA)
DATA.mkdir(parents=True)


def write_split(name, rows, split="train"):
    folder = DATA / name
    folder.mkdir(parents=True, exist_ok=True)
    Dataset.from_list(rows).to_parquet(folder / f"{split}.parquet")
    print(f"{name}/{split}: {len(rows)} rows")


TEXT_ROWS = [
    "Question: 2 + 3?\nAnswer: 5.",
    "Question: 7 * 8?\nAnswer: 56.",
    "Question: 10 - 4?\nAnswer: 6.",
    "Question: 9 + 1?\nAnswer: 10.",
]
write_split("sft_text", [{"text": text} for text in TEXT_ROWS * 4])

CHAT_ROWS = [
    [
        {"role": "system", "content": "You are a terse math tutor."},
        {"role": "user", "content": "What is 2 + 3?"},
        {"role": "assistant", "content": "5"},
    ],
    [
        {"role": "system", "content": "You are a terse math tutor."},
        {"role": "user", "content": "What is 7 * 8?"},
        {"role": "assistant", "content": "56"},
    ],
]
write_split("sft_chat", [{"messages": messages} for messages in CHAT_ROWS * 4])

write_split("dpo", [
    {"prompt": "What is 2 + 3?", "chosen": "5", "rejected": "6"},
    {"prompt": "What is 7 * 8?", "chosen": "56", "rejected": "54"},
    {"prompt": "What is 10 - 4?", "chosen": "6", "rejected": "5"},
    {"prompt": "What is 9 + 1?", "chosen": "10", "rejected": "9"},
] * 2)

write_split("grpo", [
    {"question": "What is 2 + 3? Put the final answer in \\boxed{5}.", "answer": "5"},
    {"question": "What is 7 * 8? Put the final answer in \\boxed{56}.", "answer": "56"},
    {"question": "What is 10 - 4? Put the final answer in \\boxed{6}.", "answer": "6"},
    {"question": "What is 9 + 1? Put the final answer in \\boxed{10}.", "answer": "10"},
] * 2)

write_split("custom", [{"prompt": f"What is {number} + {number}?"} for number in range(1, 9)])
write_split("custom_eval", [{"prompt": "What is 100 + 1?"}])

from dqntrain.data import load_stage_data

preview_config = dq.Data(
    str(DATA / "grpo"),
    limit=4,
    columns={"prompt": "question"},
    filter=lambda row: "What" in row["question"],
    map=lambda row: {"prompt": row["prompt"].strip()},
)
preview = load_stage_data(preview_config, "GRPO")
print("preview columns:", preview["train"].column_names)
assert preview["train"].column_names == ["prompt", "answer"]
assert len(preview["train"]) == 4

## 3. Callbacks, a subprocess-sandboxed reward, and the Custom stage

`EventRecorder` keeps its mutable state in the module-level `EVENTS` list, not on the
instance. dqntrain hashes callback attributes, so mutable attributes would change the
stage hash after a run; this pattern keeps reruns stable.

`subprocess_reward` runs each completion through a separate Python process with a
5-second timeout, the same shape as a unit-test sandbox reward. It scores `\boxed{...}`.

The Custom stage adds a second LoRA adapter (`teacher`), generates with the student,
the teacher, and the adapter-disabled base, saves an adapter checkpoint via
`ctx.save`, and removes the teacher before returning.

In [ ]:
from dqntrain.errors import ConfigChangedError, LockHeldError
from dqntrain.hub import LocalStore
from dqntrain.rewards import _text, contains_reward, format_reward
from dqntrain.state import utcnow
from transformers import TrainerCallback

import dqntrain.backend as backend

EVENTS = []


class EventRecorder(TrainerCallback):
    def __init__(self, label):
        self.label = label

    def _record(self, event, state):
        EVENTS.append((self.label, event, int(state.global_step)))

    def on_train_begin(self, args, state, control, **kwargs):
        self._record("train_begin", state)

    def on_save(self, args, state, control, **kwargs):
        self._record("save", state)

    def on_step_end(self, args, state, control, **kwargs):
        self._record("step_end", state)

    def on_train_end(self, args, state, control, **kwargs):
        self._record("train_end", state)


class CrashAfter(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if os.environ.get("SMOKE_CRASH") == "1" and state.global_step >= 2:
            raise RuntimeError("intentional crash for the resume test")


def subprocess_reward(completions, **kwargs):
    program = r"""
import re, sys
text = sys.stdin.read()
sys.exit(0 if re.search(r"\\boxed\{[^}]+\}", text) else 1)
"""
    scores = []
    for completion in completions:
        try:
            result = subprocess.run(
                [sys.executable, "-c", program],
                input=_text(completion),
                text=True,
                capture_output=True,
                timeout=5,
            )
            scores.append(1.0 if result.returncode == 0 else 0.0)
        except subprocess.TimeoutExpired:
            scores.append(0.0)
    return scores


def custom_stage(model, tokenizer, data, ctx):
    from peft import LoraConfig

    print("custom rows:", len(data["train"]), "| eval rows:", len(data["eval"]))
    model.add_adapter("teacher", LoraConfig(r=4, lora_alpha=8, target_modules="all-linear"))
    prompts = [row["prompt"] for row in data["train"]]
    student = backend.generate(model, tokenizer, prompts[:1], max_new_tokens=24, do_sample=False)
    model.set_adapter("teacher")
    teacher = backend.generate(model, tokenizer, prompts[:1], max_new_tokens=24, do_sample=False)
    with model.disable_adapter():
        base = backend.generate(model, tokenizer, prompts[:1], max_new_tokens=24, do_sample=False)
    model.set_adapter("default")
    print("adapters while switched:", list(model.peft_config))
    model.delete_adapter("teacher")
    print("adapters after delete:", list(model.peft_config))
    print("student:", student[0][:60])
    print("teacher:", teacher[0][:60])
    print("base:   ", base[0][:60])

    adapter_dir = os.path.join(ctx.run_dir, "custom-adapter")
    backend.save_adapter(model, adapter_dir)
    ctx.save(1, adapter_dir)
    ctx.log({"event": "custom_adapter_switch", "adapters": list(model.peft_config)})
    return model

## 4. Main pipeline

Seven short stages, all on one 4-bit fp16 model: SFT text, SFT chat, SFT template,
DPO, GRPO, the Custom adapter demo, and an fp16 merge. `summary()` parses the data and
hashes every stage without loading a model.

In [ ]:
BASE = "Qwen/Qwen3-0.6B"
DTYPE = "fp16"
RUN_NAME = f"{TAG}-main"
STORE = LocalStore(str(RUNS))

LOAD = dq.Load(four_bit=True, dtype=DTYPE, max_seq_length=1024)
LORA = dq.LoRA(r=8, alpha=16, dropout=0.0)
recorder = EventRecorder("main")

stages = [
    dq.SFT(
        "sft-text",
        data=dq.Data(str(DATA / "sft_text"), limit=8),
        lora=LORA,
        train=dq.Train(lr=2e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
        save_every=1,
        format="text",
        callbacks=[recorder],
    ),
    dq.SFT(
        "sft-chat",
        data=dq.Data(str(DATA / "sft_chat"), limit=8),
        lora=LORA,
        train=dq.Train(lr=2e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
        save_every=1,
        format="chatml",
        callbacks=[recorder],
    ),
    dq.SFT(
        "sft-template",
        data=dq.Data(str(DATA / "sft_chat"), limit=8),
        lora=LORA,
        train=dq.Train(lr=2e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
        save_every=1,
        format="template",
    ),
    dq.DPO(
        "dpo",
        data=dq.Data(str(DATA / "dpo"), limit=8),
        lora=LORA,
        train=dq.Train(lr=1e-5, batch=1, grad_accum=2, max_steps=1, beta=0.1,
                       max_length=512, logging_steps=1, report_to=[]),
    ),
    dq.GRPO(
        "grpo",
        data=dq.Data(
            str(DATA / "grpo"),
            limit=8,
            columns={"prompt": "question"},
            filter=lambda row: len(row["question"]) > 0,
            map=lambda row: {"prompt": row["prompt"].strip()},
        ),
        reward_funcs=[format_reward(r"\\boxed\{.*\}"), contains_reward("\\boxed"), subprocess_reward],
        reward_weights=[0.2, 0.6, 0.2],
        lora=LORA,
        train=dq.Train(lr=1e-5, batch=2, generations=2, max_completion_length=64,
                       max_steps=1, logging_steps=1, report_to=[]),
    ),
    dq.Custom(
        "mopd-switch",
        fn=custom_stage,
        data=dq.Data(str(DATA / "custom"), limit=8,
                     eval=dq.Data(str(DATA / "custom_eval"), limit=1)),
        save_every=1,
    ),
    dq.Merge("merge", dtype=DTYPE),
]
if RUN_EVAL:
    stages.append(dq.Eval("eval-arc-easy", tasks=["arc_easy"], limit=10))

pipe = dq.Sequential(
    RUN_NAME,
    base=BASE,
    load=LOAD,
    stages=stages,
    store=STORE,
    cache_dir=str(CACHE),
)
pipe.summary()

In [ ]:
pipe.run()
pipe.status()
print("callback events:", EVENTS)

## 5. Verify the artifacts

Checks every stage finished, callbacks fired, checkpoint metadata records the backend
versions, the custom checkpoint and log exist, and the merged fp16 model reloads and
generates.

In [ ]:
manifest = STORE.read_manifest(RUN_NAME)
records = {record["name"]: record for record in manifest["stages"]}
expected = [stage.name for stage in stages]
assert list(records) == expected
assert all(record["status"] == "done" for record in records.values()), records
assert records["sft-text"]["checkpoints"][-1]["step"] == 1
assert len(records["mopd-switch"]["checkpoints"]) == 1
assert {"train_begin", "save", "step_end", "train_end"} <= {event for _, event, _ in EVENTS}

for stage_name in expected:
    assert (RUNS / RUN_NAME / stage_name / "final").is_dir(), stage_name

meta = json.loads((RUNS / RUN_NAME / "sft-text" / "ckpt-000001" / "meta.json").read_text())
for key in ("torch", "transformers", "trl", "peft", "dqntrain", "base", "config_hash"):
    assert key in meta, key
assert meta["base"] == BASE
print("checkpoint meta versions:", {key: meta[key] for key in ("torch", "transformers", "trl", "peft")})

adapter_config = json.loads((RUNS / RUN_NAME / "sft-chat" / "final" / "adapter_config.json").read_text())
assert adapter_config["r"] == 8
assert (RUNS / RUN_NAME / "sft-text" / "final" / "adapter_config.json").is_file()

custom_checkpoint = RUNS / RUN_NAME / "mopd-switch" / "ckpt-000001"
assert (custom_checkpoint / "adapter_config.json").is_file()
assert (custom_checkpoint / "adapter_model.safetensors").is_file()
custom_log = (RUNS / RUN_NAME / "logs" / "mopd-switch.jsonl").read_text()
assert "custom_adapter_switch" in custom_log

merged_dir = RUNS / RUN_NAME / "merge" / "final"
assert (merged_dir / "config.json").is_file()
assert not (merged_dir / "adapter_config.json").exists()
merged_model, merged_tokenizer = backend.load_model(str(merged_dir), dtype=DTYPE, max_seq_length=512)
sample = backend.generate(merged_model, merged_tokenizer, ["What is 2 + 3?"],
                          max_new_tokens=16, do_sample=False)[0]
print("merged sample:", sample[:80])
assert isinstance(sample, str)
del merged_model

hashes = [record["config_hash"] for record in manifest["stages"]]
assert len(set(hashes)) == len(hashes), hashes
print("main pipeline checks passed")

## 6. Kill and resume

The callback raises at step 2 while `SMOKE_CRASH=1`. Step 1 was already uploaded, so
the second run resumes from `ckpt-000001` and finishes at step 3. The callback itself
is unchanged, so the config hash stays stable; only the environment variable changes.

In [ ]:
RESUME_NAME = f"{TAG}-resume"
resume_pipe = dq.Sequential(
    RESUME_NAME,
    base=BASE,
    load=LOAD,
    store=STORE,
    cache_dir=str(CACHE),
    stages=[
        dq.SFT(
            "sft-resume",
            data=dq.Data(str(DATA / "sft_text"), limit=8),
            lora=LORA,
            train=dq.Train(lr=2e-4, batch=2, max_steps=3, logging_steps=1, report_to=[]),
            save_every=1,
            format="text",
            callbacks=[CrashAfter()],
        )
    ],
)

os.environ["SMOKE_CRASH"] = "1"
try:
    resume_pipe.run()
except RuntimeError as error:
    print("crashed on purpose:", error)
else:
    raise AssertionError("the crash callback did not fire")
resume_pipe.status()
assert STORE.read_manifest(RESUME_NAME)["stages"][0]["status"] == "running"

os.environ.pop("SMOKE_CRASH")
resume_pipe.run()
resume_pipe.status()
resume_record = STORE.read_manifest(RESUME_NAME)["stages"][0]
assert resume_record["status"] == "done"
assert [item["step"] for item in resume_record["checkpoints"]] == [1, 2, 3]
print("resume check passed: ckpt-000001 -> done at step 3")

## 7. Config hashing and `rerun()`

Changing a callback changes the stage hash, and so does changing `lr`. A live config
change is blocked until `rerun()` marks that stage and everything after it as pending.

In [ ]:
HASH_NAME = f"{TAG}-hash"
hash_pipe = dq.Sequential(
    HASH_NAME,
    base=BASE,
    load=LOAD,
    store=STORE,
    cache_dir=str(CACHE),
    stages=[
        dq.SFT(
            "sft-hash",
            data=dq.Data(str(DATA / "sft_text"), limit=8),
            lora=LORA,
            train=dq.Train(lr=1e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
            save_every=1,
            format="text",
        )
    ],
)
hash_pipe.run()
first_hash = hash_pipe._hashes()[0]["hash"]

hash_pipe.stages[0].callbacks.append(EventRecorder("hash-extra"))
assert hash_pipe._hashes()[0]["hash"] != first_hash
hash_pipe.stages[0].callbacks = []
print("callback hash check passed")

hash_pipe.stages[0].train.values["learning_rate"] = 2e-4
try:
    hash_pipe.run()
except ConfigChangedError as error:
    print("config change blocked:", error)
else:
    raise AssertionError("a changed config was not blocked")

hash_pipe.rerun("sft-hash")
hash_pipe.run()
hash_pipe.status()
hash_record = STORE.read_manifest(HASH_NAME)["stages"][0]
assert hash_record["status"] == "done"
assert hash_record["config_hash"] != first_hash
print("rerun check passed:", first_hash, "->", hash_record["config_hash"])

## 8. Locks

A fresh heartbeat from another holder must block `run()`. A stale heartbeat (older
than 15 minutes) is taken over, and the takeover is written to `logs/_pipeline.jsonl`.

In [ ]:
LOCK_NAME = f"{TAG}-lock"
lock_pipe = dq.Sequential(
    LOCK_NAME,
    base=BASE,
    load=LOAD,
    store=STORE,
    cache_dir=str(CACHE),
    stages=[
        dq.SFT(
            "sft-lock",
            data=dq.Data(str(DATA / "sft_text"), limit=8),
            lora=LORA,
            train=dq.Train(lr=1e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
            save_every=1,
            format="text",
        )
    ],
)

STORE.write_manifest(
    LOCK_NAME,
    {"version": 1, "stages": [], "lock": {"holder": "ghost-notebook", "heartbeat": utcnow()}},
)
try:
    lock_pipe.run()
except LockHeldError as error:
    print("fresh lock rejected:", error)
else:
    raise AssertionError("a fresh lock was not rejected")

STORE.write_manifest(
    LOCK_NAME,
    {"version": 1, "stages": [], "lock": {"holder": "ghost-notebook", "heartbeat": "2000-01-01T00:00:00Z"}},
)
lock_pipe.run()
lock_manifest = STORE.read_manifest(LOCK_NAME)
assert lock_manifest["stages"][0]["status"] == "done"
assert not lock_manifest["lock"].get("holder")
lock_log = (STORE.read_bytes(LOCK_NAME, "logs/_pipeline.jsonl") or b"").decode()
assert "lock_takeover" in lock_log
print("lock check passed: fresh rejected, stale taken over")

## 9. Optional Hub round-trip

Set `RUN_HUB=1` and `HF_TOKEN` to upload a one-step run to a new private repo and
verify the manifest and checkpoint arrived. The repo is left in place for inspection;
delete it manually if you do not need it.

In [ ]:
if not RUN_HUB:
    print("RUN_HUB is off; set RUN_HUB=1 and HF_TOKEN to test HFStore uploads")
elif not os.environ.get("HF_TOKEN"):
    print("HF_TOKEN is missing; skipping the Hub round-trip")
else:
    from huggingface_hub import HfApi

    api = HfApi(token=os.environ["HF_TOKEN"])
    user = api.whoami()["name"]
    repo = f"{user}/dqntrain-t4-smoke-{TAG}"
    hub_name = f"{TAG}-hub"
    hub_pipe = dq.Sequential(
        hub_name,
        repo=repo,
        base=BASE,
        load=LOAD,
        full_state=False,
        stages=[
            dq.SFT(
                "sft-hub",
                data=dq.Data(str(DATA / "sft_text"), limit=8),
                lora=LORA,
                train=dq.Train(lr=1e-4, batch=2, max_steps=1, logging_steps=1, report_to=[]),
                save_every=1,
                format="text",
            )
        ],
    )
    hub_pipe.run()
    files = api.list_repo_files(repo)
    assert f"{hub_name}/dqntrain.json" in files
    assert any(path.startswith(f"{hub_name}/sft-hub/ckpt-") for path in files)
    print("hub round-trip passed:", len(files), "files in", repo)

## 10. Result

If no cell raised, the smoke test passed: model loading (4-bit fp16), all three SFT
format modes, DPO, GRPO with a subprocess reward, callbacks, custom multi-adapter
generation, adapter save/checkpointing, merge + reload, metadata, crash/resume,
config hashing/rerun, and lock behavior all worked on the T4.

Not covered here: `Eval` unless `RUN_EVAL=1`, Hub unless `RUN_HUB=1`, multi-GPU
(distributed training is out of scope; dqntrain uses one GPU per process), and vLLM.

In [ ]:
print("smoke test passed")
print("runs:", RUNS.resolve())
print("main:", RUN_NAME)
print("resume:", RESUME_NAME)
print("hash:", HASH_NAME)
print("lock:", LOCK_NAME)
print("cache:", CACHE.resolve())

for path in sorted(RUNS.glob(f"{TAG}-*")):
    size = sum(item.stat().st_size for item in path.rglob("*") if item.is_file())
    print(f"  {path.name:<24} {size / 1e6:8.2f} MB")